# DeepCDR on a small subset (Colab)

Runs [DeepCDR](https://github.com/kimmo1019/DeepCDR) (hybrid graph conv net: drug graph + mutation + expression + methylation) on a random subset of **80** GDSC/CCLE (cell line, drug) pairs, then reports overall and **per-sex** Pearson correlation on held-out pairs.

**Why not just run `run_DeepCDR.py`?** The original code is Python 2 / Keras 2.1.4 / TF 1.13 and will not run on Colab. This notebook keeps the same data processing and architecture but ports the model to current Keras (a small custom `GraphConv` layer replaces `layers/graph.py`).

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (CPU also works with the default small subset, just slower).

Knobs live in the *Settings* cell: `N_SAMPLES`, `EPOCHS`, `USE_MUT / USE_GEXP / USE_METHY`.

## 1. Get the repo + data

In [ ]:
import os
if not os.path.exists('DeepCDR/data/CCLE'):
    # Option A (default): clone the public repo (includes the preprocessed data, ~30 MB)
    !git clone --depth 1 https://github.com/kimmo1019/DeepCDR.git
    # Option B: if you'd rather use your own copy, upload DeepCDR.zip in the Files sidebar and run:
    # !unzip -q DeepCDR.zip

if not os.path.exists('DeepCDR/data/GDSC/drug_graph_feat'):
    !cd DeepCDR/data/GDSC && unzip -q -o drug_graph_feat.zip

!ls DeepCDR/data/CCLE DeepCDR/data/GDSC | head -30

In [ ]:
!pip install -q hickle
import tensorflow as tf, keras, hickle
print('TF', tf.__version__, '| Keras', keras.__version__, '| hickle', hickle.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Settings

In [ ]:
SEED       = 0
N_SAMPLES  = 80       # (cell line, drug) pairs to use (tiny smoke-test subset)
EPOCHS     = 40
BATCH_SIZE = 16
PATIENCE   = 10       # early stopping on validation Pearson
USE_MUT, USE_GEXP, USE_METHY = True, True, True
UNIT_LIST  = [256, 256, 256]   # GCN layer widths (same as paper)
MAX_ATOMS  = 100

DPATH = 'DeepCDR/data'
Drug_info_file          = f'{DPATH}/GDSC/1.Drug_listMon Jun 24 09_00_55 2019.csv'
Cell_line_info_file     = f'{DPATH}/CCLE/Cell_lines_annotations_20181226.txt'
Drug_feature_dir        = f'{DPATH}/GDSC/drug_graph_feat'
Genomic_mutation_file   = f'{DPATH}/CCLE/genomic_mutation_34673_demap_features.csv'
Cancer_response_file    = f'{DPATH}/CCLE/GDSC_IC50.csv'
Gene_expression_file    = f'{DPATH}/CCLE/genomic_expression_561celllines_697genes_demap_features.csv'
Methylation_file        = f'{DPATH}/CCLE/genomic_methylation_561celllines_808genes_demap_features.csv'

## 3. Load tables and build the (cell line, drug, ln IC50) list
Same logic as `MetadataGenerate` in the original script, vectorized so it doesn't take minutes. Also pulls the `Gender` column from the CCLE annotation file so we can look at per-sex performance.

In [ ]:
import csv, random
import numpy as np, pandas as pd
import hickle as hkl
from scipy.stats import pearsonr

random.seed(SEED); np.random.seed(SEED)

# drug GDSC id -> PubChem id
with open(Drug_info_file) as f:
    rows = list(csv.reader(f))
drugid2pubchemid = {r[0]: r[5] for r in rows if len(r) > 5 and r[5].isdigit()}

# cell line -> (cancer type, sex)   (cols: 1=depMapID, 12=Gender, last=tcga_code)
ann = pd.read_csv(Cell_line_info_file, sep='\t')
cellline2type = dict(zip(ann['depMapID'], ann.iloc[:, -1]))
cellline2sex  = dict(zip(ann['depMapID'], ann['Gender']))

# omics
mutation_feature    = pd.read_csv(Genomic_mutation_file, index_col=0)
gexpr_feature       = pd.read_csv(Gene_expression_file,  index_col=0)
methylation_feature = pd.read_csv(Methylation_file,      index_col=0)
mutation_feature = mutation_feature.loc[gexpr_feature.index]       # keep overlapping lines
assert mutation_feature.shape[0] == gexpr_feature.shape[0] == methylation_feature.shape[0]
print('mutation', mutation_feature.shape, '| expression', gexpr_feature.shape, '| methylation', methylation_feature.shape)

# drug graphs (hickle files written by hickle 3.4.5 / py3.6; modern hickle reads them via its legacy loader)
drug_feature = {}
for fn in os.listdir(Drug_feature_dir):
    feat_mat, adj_list, _ = hkl.load(f'{Drug_feature_dir}/{fn}')
    drug_feature[fn.split('.')[0]] = (np.asarray(feat_mat), adj_list)
print('drugs with graphs:', len(drug_feature))

# IC50 matrix (rows = "<name>:<gdsc_id>", cols = cell lines) -> long format
ic50 = pd.read_csv(Cancer_response_file, index_col=0)
ic50 = ic50.loc[[i for i in ic50.index if i.split(':')[1] in drugid2pubchemid]]
long = ic50.stack().reset_index()
long.columns = ['drug_row', 'cellline', 'ln_ic50']
long['pubchem'] = long['drug_row'].map(lambda s: drugid2pubchemid[s.split(':')[-1]])
long = long[long['pubchem'].isin(drug_feature) &
            long['cellline'].isin(mutation_feature.index) &
            long['cellline'].isin(cellline2type)].copy()
long['cancer_type'] = long['cellline'].map(cellline2type)
long['sex']         = long['cellline'].map(cellline2sex)
print(f"{len(long)} instances | {long.cellline.nunique()} cell lines | {long.pubchem.nunique()} drugs")
print(long['sex'].value_counts(dropna=False))

## 4. Take a small subset and split
Random sample of `N_SAMPLES` (80) pairs, split 80/10/10 into train/val/test (64/8/8). This is a plain random split of pairs (like the original paper's setup, not an unseen-cell-line or unseen-drug split).

In [ ]:
sub = long.sample(n=min(N_SAMPLES, len(long)), random_state=SEED).reset_index(drop=True)
idx = np.random.RandomState(SEED).permutation(len(sub))
n_tr, n_va = int(0.8*len(sub)), int(0.1*len(sub))
splits = {'train': sub.iloc[idx[:n_tr]], 'val': sub.iloc[idx[n_tr:n_tr+n_va]], 'test': sub.iloc[idx[n_tr+n_va:]]}
for k, v in splits.items():
    print(k, len(v), '| cell lines', v.cellline.nunique(), '| drugs', v.pubchem.nunique())

## 5. Featurize (drug graph padding + adjacency normalization, same as original)

In [ ]:
def normalize_adj(adj):
    adj = adj + np.eye(adj.shape[0])
    d = np.power(adj.sum(1), -0.5)
    return adj * d[:, None] * d[None, :]          # D^-1/2 (A+I) D^-1/2

_graph_cache = {}
def graph_feat(pubchem_id):
    if pubchem_id in _graph_cache: return _graph_cache[pubchem_id]
    feat_mat, adj_list = drug_feature[str(pubchem_id)]
    n = feat_mat.shape[0]
    feat = np.zeros((MAX_ATOMS, feat_mat.shape[-1]), dtype='float32'); feat[:n] = feat_mat
    adj = np.zeros((MAX_ATOMS, MAX_ATOMS), dtype='float32')
    for i, nbrs in enumerate(adj_list):
        for j in nbrs: adj[i, int(j)] = 1
    assert np.allclose(adj, adj.T)
    a1 = adj[:n, :n]; a2 = adj[n:, n:]             # real atoms / padding block, normalized separately (as in original)
    adj[:n, :n] = normalize_adj(a1)
    if a2.shape[0] > 0: adj[n:, n:] = normalize_adj(a2)
    _graph_cache[pubchem_id] = (feat, adj)
    return feat, adj

def featurize(df):
    drug_x = np.stack([graph_feat(p)[0] for p in df.pubchem])
    drug_a = np.stack([graph_feat(p)[1] for p in df.pubchem])
    mut  = mutation_feature.loc[df.cellline].values.astype('float32')[:, None, :, None]
    gexp = gexpr_feature.loc[df.cellline].values.astype('float32')
    meth = methylation_feature.loc[df.cellline].values.astype('float32')
    y    = df.ln_ic50.values.astype('float32')
    return [drug_x, drug_a, mut, gexp, meth], y

X_tr, y_tr = featurize(splits['train'])
X_va, y_va = featurize(splits['val'])
X_te, y_te = featurize(splits['test'])
print([a.shape for a in X_tr])

## 6. Model (port of `prog/model.py` + `prog/layers/graph.py` to current Keras)
Architecture is unchanged: 4 GraphConv layers on the drug (256-256-256-100, ReLU + BN + dropout, global max pool), a conv branch for mutations, dense branches for expression and methylation, concatenate, dense(300), then three conv/pool blocks and a linear output.

In [ ]:
import keras
from keras import layers, ops

@keras.saving.register_keras_serializable(package='deepcdr')
class GraphConv(layers.Layer):
    """out = A^T (X W + b)   (A is the pre-normalized adjacency, as in the original layer)"""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs); self.units = units
    def build(self, input_shape):
        d = input_shape[0][-1]
        self.W = self.add_weight(shape=(d, self.units), initializer='glorot_uniform', name='W')
        self.b = self.add_weight(shape=(self.units,), initializer='zeros', name='b')
    def call(self, inputs):
        x, a = inputs
        h = ops.matmul(x, self.W) + self.b
        return ops.matmul(ops.transpose(a, (0, 2, 1)), h)
    def compute_output_shape(self, input_shape):
        return (*input_shape[0][:2], self.units)
    def get_config(self):
        return {**super().get_config(), 'units': self.units}

def gcn_block(x, adj, units):
    x = GraphConv(units)([x, adj])
    x = layers.Activation('relu')(x)
    x = layers.BatchNormalization()(x)
    return layers.Dropout(0.1)(x)

def build_deepcdr(drug_dim, mut_dim, gexp_dim, meth_dim, units_list=UNIT_LIST,
                  use_mut=True, use_gexp=True, use_methy=True):
    drug_x = layers.Input((MAX_ATOMS, drug_dim), name='drug_feat_input')
    drug_a = layers.Input((MAX_ATOMS, MAX_ATOMS), name='drug_adj_input')
    mut_in = layers.Input((1, mut_dim, 1), name='mutation_feat_input')
    gexp_in = layers.Input((gexp_dim,), name='gexpr_feat_input')
    meth_in = layers.Input((meth_dim,), name='methy_feat_input')

    g = drug_x
    for u in units_list: g = gcn_block(g, drug_a, u)
    g = gcn_block(g, drug_a, 100)
    x_drug = layers.GlobalMaxPooling1D()(g)

    m = layers.Conv2D(50, (1, 700), strides=(1, 5), activation='tanh')(mut_in)
    m = layers.MaxPooling2D((1, 5))(m)
    m = layers.Conv2D(30, (1, 5), strides=(1, 2), activation='relu')(m)
    m = layers.MaxPooling2D((1, 10))(m)
    m = layers.Flatten()(m)
    x_mut = layers.Dropout(0.1)(layers.Dense(100, activation='relu')(m))

    def dense_branch(inp):
        h = layers.Dense(256)(inp); h = layers.Activation('tanh')(h)
        h = layers.BatchNormalization()(h); h = layers.Dropout(0.1)(h)
        return layers.Dense(100, activation='relu')(h)
    x_gexp, x_meth = dense_branch(gexp_in), dense_branch(meth_in)

    parts = [x_drug]
    if use_mut:   parts.append(x_mut)
    if use_gexp:  parts.append(x_gexp)
    if use_methy: parts.append(x_meth)
    x = layers.Concatenate()(parts) if len(parts) > 1 else parts[0]
    x = layers.Dropout(0.1)(layers.Dense(300, activation='tanh')(x))
    x = layers.Reshape((1, 300, 1))(x)
    x = layers.Conv2D(30, (1, 150), activation='relu')(x);  x = layers.MaxPooling2D((1, 2))(x)
    x = layers.Conv2D(10, (1, 5),  activation='relu')(x);   x = layers.MaxPooling2D((1, 3))(x)
    x = layers.Conv2D(5,  (1, 5),  activation='relu')(x);   x = layers.MaxPooling2D((1, 3))(x)
    x = layers.Dropout(0.1)(x); x = layers.Flatten()(x); x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, name='output')(x)
    return keras.Model([drug_x, drug_a, mut_in, gexp_in, meth_in], out)

keras.utils.set_random_seed(SEED)
model = build_deepcdr(X_tr[0].shape[-1], X_tr[2].shape[-2], X_tr[3].shape[-1], X_tr[4].shape[-1],
                      use_mut=USE_MUT, use_gexp=USE_GEXP, use_methy=USE_METHY)
model.summary(line_length=100)

## 7. Train (early stopping on validation Pearson, like the original `MyCallback`)

In [ ]:
class PearsonEarlyStop(keras.callbacks.Callback):
    def __init__(self, X, y, patience):
        super().__init__(); self.X, self.y, self.patience = X, y, patience
    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_w = -np.inf, 0, None
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.X, verbose=0)[:, 0]
        r = pearsonr(self.y, p)[0]
        logs['val_pcc'] = r
        print(f'   val PCC = {r:.4f}')
        if r > self.best:
            self.best, self.wait, self.best_w = r, 0, self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience: self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
hist = model.fit(X_tr, y_tr, batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2,
                 callbacks=[PearsonEarlyStop(X_va, y_va, PATIENCE)])

## 8. Evaluate: overall and per-sex

In [ ]:
pred = model.predict(X_te, verbose=0)[:, 0]
te = splits['test'].copy(); te['pred'] = pred

def report(df, name):
    if len(df) < 3: print(f'{name:<10} n={len(df):>4}   (too few)'); return
    r = pearsonr(df.pred, df.ln_ic50)[0]
    rmse = float(np.sqrt(((df.pred - df.ln_ic50) ** 2).mean()))
    print(f'{name:<10} n={len(df):>4}   Pearson={r:.3f}   RMSE={rmse:.3f}')

report(te, 'overall')
for s, d in te.groupby(te['sex'].fillna('missing')):
    report(d, str(s))
print('\nNote: with only 80 pairs (about 8 test pairs) all metrics, especially per-sex, are essentially noise; this run only checks that the pipeline works end to end.')

te.to_csv('deepcdr_subset_test_predictions.csv', index=False)   # keep predictions for per-sex metrics later
model.save('deepcdr_subset.keras')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for s, d in te.groupby(te['sex'].fillna('missing')):
    ax.scatter(d.ln_ic50, d.pred, s=10, alpha=.6, label=f'{s} (n={len(d)})')
lo, hi = te.ln_ic50.min(), te.ln_ic50.max()
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('true ln(IC50)'); ax.set_ylabel('predicted'); ax.legend(); plt.tight_layout(); plt.show()

## Notes
- **Random pair split:** the same cell line can appear in train and test (with different drugs). Your project plan calls for unseen-cell-line / drug / tissue splits; swap the split in section 4 for that (group by `cellline`, `pubchem`, or `cancer_type`).
- **Sex labels:** `Gender` comes straight from the CCLE annotation file. This is exactly the metadata your proposal wants to audit (missing/mislabeled), so expect some `NaN`s.
- **Scaling up:** raise `N_SAMPLES` (e.g. 3000) once this runs; RAM is dominated by the mutation matrix (~139 KB per pair). If Colab runs out of memory, build batches lazily (a `keras.utils.PyDataset`) instead of featurizing everything up front.
- **Differences from the original:** TF1/Keras 2.1 -> current Keras; Python 2 `print` removed; fixed-count `ModelCheckpoint` replaced with best-weights restore; early stopping uses a held-out validation split instead of the test set (the original validated on the test set).